# VAIC2026 — e5 lan 2: cau hinh THAN TRONG

Lan 1 (lr 1e-4, 10 epoch, 4 hard neg) cho ket qua **am**:
R@1 +1 cau nhung R@5 **-1 cau**, MRR dung yen (0.8210 -> 0.8207), nDCG xau di.
Tuc la khong cai thien xep hang, chi xao cho.

Gia thuyet: 46 vi du voi lr 1e-4 la qua manh cho khong gian embedding da
pretrain tot. Cross-encoder chiu duoc vi cham tung cap; bi-encoder co khong
gian TOAN CUC nen meo la hong ca recall.

Lan nay: **lr 2e-5** (thap hon 5 lan), **4 epoch**, **8 hard negative**.
Neu van khong thang duoc e5 goc o R@5 thi ket luan la: o quy mo du lieu nay,
KHONG nen fine-tune tang 1 -- phai sua bang du lieu hoac chunking.

In [ ]:
# Kaggle's torch (2.10+cu128) dropped Pascal sm_60 but the assigned GPU is often a
# P100 (sm_60) -> "no kernel image". Install a torch that supports P100 AND T4.
# MUST run before any `import torch`. Also drop torchvision/torchaudio (mismatched
# -> "torchvision::nms does not exist") and torchao (peft rejects Kaggle's 0.10.0).
!pip uninstall -y -q torchvision torchaudio torchao
!pip install -q rank-bm25 peft
!pip install -q torch==2.6.0+cu124 --extra-index-url https://download.pytorch.org/whl/cu124

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda)
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0), '| sm_', torch.cuda.get_device_capability(0))
    x = torch.randn(8, device='cuda'); print('CUDA op OK:', float((x + 1).sum()))

In [ ]:
import glob

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'{name} not found under /kaggle/input'
    return hits[0]

TRAIN_PY = find('train_retriever.py')
EVAL_PY  = find('eval_reranker.py')
CHUNKS   = find('chunks.jsonl')
EVAL_33  = find('eval_dienbien_v2.jsonl')
TRAIN_A  = find('train_dienbien_v2.jsonl')
OUT_E5   = '/kaggle/working/e5-conservative'
for k in ['TRAIN_PY','EVAL_PY','CHUNKS','EVAL_33','TRAIN_A']:
    print(k, '=', eval(k))
for k, p in [('TRAIN_A', TRAIN_A), ('EVAL_33', EVAL_33)]:
    print(k, 'rows =', sum(1 for l in open(p, encoding='utf-8') if l.strip()))

## Train e5 — lr thap, it epoch, nhieu hard negative

In [ ]:
!python {TRAIN_PY}     --train {TRAIN_A} --out {OUT_E5}     --base-model intfloat/multilingual-e5-large     --use-lora --lora-r 16 --lora-alpha 32     --epochs 4 --batch-size 4 --grad-accum 2 --lr 2e-5     --max-len 320 --n-neg 8 --num-workers 2

## Eval tren 33 cau dong bang

In [ ]:
!python {EVAL_PY} --chunks {CHUNKS} --eval {EVAL_33}     --e5-model intfloat/multilingual-e5-large     --e5-finetuned {OUT_E5}     --base-model BAAI/bge-reranker-v2-m3     --finetuned /kaggle/working/__none__
import shutil; shutil.copy('eval_results.json', '/kaggle/working/eval_e5_conservative.json')

In [ ]:
import json
r = json.load(open('/kaggle/working/eval_e5_conservative.json'))
base = r.get('e5 retrieval only', {})
ft = r.get('e5 fine-tuned (retrieval only)', {})
print(json.dumps(r, indent=2, ensure_ascii=False))
if base and ft:
    print('=== TANG 1: e5 goc -> e5 fine-tune conservative (33 cau) ===')
    for k in ['Recall@1','Recall@3','Recall@5','MRR','nDCG@5']:
        d = ft[k] - base[k]
        print('  %-9s %.4f -> %.4f   (%+.4f = %+.1f cau)' % (k, base[k], ft[k], d, d*33))
